In [1]:
import os
import sys
import numpy as np
import copy as cp
import math
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D

import scipy.stats
import pickle

In [2]:
sys.path.append(os.path.abspath('../code/'))

from HOC_amplitude_functions import compute_amp_spectrum_beneficial, compute_amp_spectrum_beneficial_better
from HOC_amplitude_functions import compute_amp_spectrum_adaptive, compute_amp_spectrum_adaptive_better
from path_approximation import compute_expected_paths
from moment_approximations import compute_Ma
from HOC_amplitude_functions import compute_amp_spectrum_adaptive_better

In [3]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.titlesize'] = 8    # Size of the plot title
plt.rcParams['axes.labelsize'] = 8    # Size of x and y labels
plt.rcParams['xtick.labelsize'] = 7   # Size of x-tick labels
plt.rcParams['ytick.labelsize'] = 7   # Size of y-tick labels
plt.rcParams['legend.fontsize'] = 7
plt.rcParams['font.size'] = 8

plt.rcParams.update({
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'xtick.minor.size': 2,
    'ytick.minor.size': 2,
    'xtick.major.pad': 2,
    'ytick.major.pad': 2,
    'axes.linewidth': 0.8,
})

cm_to_inch = 2.54

In [4]:
# colors for different scenarios
scenarios =  ['adaptive', 'beneficial', 'deleterious', 'random', 'conditional']
mycolors   = ['salmon', 'gold', 'cornflowerblue', 'lightgray', 'dimgray']
dark_colors = ['maroon','orange','navy','dimgray','black']

color_dict = dict (zip (scenarios, mycolors))

In [5]:
save = True

# directory with simulation results
inputdir = outputdir = '../results/HOC/2026-09-27/'

# parameters
Ms = [2,4,6]
L  = 100
variance = mean = 4

In [6]:
# read in simulation results
R2_list = list ()
W_list = list ()
P_list = list ()
traj_list = list ()

for m in Ms :
    R2_list.append (pickle.load ( open (os.path.join (inputdir, 'R2_' + str (m) + '.pkl'), 'rb')))
    W_list.append (pickle.load ( open (os.path.join (inputdir, 'nmax_' + str (m) + '.pkl'), 'rb')))
    P_list.append (np.loadtxt ( os.path.join (inputdir, 'npaths_' + str (m) + '.txt')))
    traj_list.append (np.loadtxt ( os.path.join (inputdir, 'trajectory_' + str (m) + '.txt')))

In [7]:
plt.rcParams['figure.figsize'] = (17 / cm_to_inch, 3.5)

fig, axs = plt.subplots (len (Ms), 5, constrained_layout=True)

i = 0
for m in Ms :
    if i == 0 :
        ntypes, nsims = P_list[0].shape
    ax  = axs[i,0]
    ax1 = axs[i,1]
    for t in range (ntypes) :
        ax.errorbar (np.arange (0, m+1, 1), np.mean (R2_list[i][t,:,:], axis=0), 
                      1.96 * np.sqrt (np.var (R2_list[i][t,:,:], axis=0) / nsims), capsize=5, markersize=4,
                      color=color_dict[scenarios[t]], marker='o')
        ax1.errorbar (np.arange (0, m+1, 1), np.append ([0], np.diff (np.mean (R2_list[i][t,:,:], axis=0), axis=0)), 
                      1.96 * np.sqrt (np.var (R2_list[i][t,:,:], axis=0) / nsims), capsize=5, markersize=4,
                      color=color_dict[scenarios[t]], marker='o')

    amp_list = list ()
    binom_coefficients = np.array ([math.comb (m, i) for i in range (m+1)])
    binom_coefficients[0] = 0
    amp_list.append (binom_coefficients)
    amp_list.append (compute_amp_spectrum_beneficial (m, mean, np.sqrt (variance), mean=False))
    amp_list.append (compute_amp_spectrum_adaptive (m, mean, np.sqrt (variance)))

    adap_spectrum = compute_amp_spectrum_adaptive_better (m, np.mean (traj_list[i], axis=0)[1:], 
                                                             np.sqrt (np.var (traj_list[i], axis=0))[1:],
                                                             C=np.cov (traj_list[i]), mean=False)

    bene_spectrum = compute_amp_spectrum_beneficial_better (m, L=L)

    amp_colors = ['black','orange','maroon']
    for a in range (len (amp_list)) :
        ax.plot (np.arange (0, m+1, 1), np.cumsum (amp_list[a]) / np.sum (amp_list[a]),
                  linestyle='dotted', color=amp_colors[a], zorder=10, linewidth=1)
        ax1.plot (np.arange (0, m+1, 1), amp_list[a] / np.sum (amp_list[a]),
                  linestyle='dotted', color=amp_colors[a], zorder=10, linewidth=1)
    
    ax.plot (np.arange (0, m+1, 1), np.cumsum (adap_spectrum) / np.sum (adap_spectrum),
        linestyle='-', color='maroon', markersize=5, marker='x', zorder=10, linewidth=.5)
    ax.plot (np.arange (0, m+1, 1), np.cumsum (bene_spectrum) / np.sum (bene_spectrum),
        linestyle='-', color='orange', markersize=5, marker='x', zorder=10, linewidth=.5)

    ax1.plot (np.arange (0, m+1, 1), adap_spectrum / np.sum (adap_spectrum),
        linestyle='-', color='maroon', markersize=5, marker='x', zorder=10, linewidth=.5)
    ax1.plot (np.arange (0, m+1, 1), bene_spectrum / np.sum (bene_spectrum),
        linestyle='-', color='orange', markersize=5, marker='x', zorder=10, linewidth=.5)
    
    ax.set_xticks ( np.arange (0, m+1, 1), np.arange (0, m+1, 1))
    ax1.set_xticks ( np.arange (0, m+1, 1), np.arange (0, m+1, 1))

    ax.set_ylabel (r'$R^2_n$')
    ax1.set_ylabel (r'$\tilde A_m$')

    if i == len (Ms) -1 :
        ax.set_xlabel (r'order, $n$')
        ax1.set_xlabel (r'order, $m$')

    if i == 0 :
        ax.set_title ('A', weight='bold', loc='left', pad=3)
        ax1.set_title ('B', weight='bold', loc='left', pad=3)

    # n maxima
    ax = axs[i,2]

    for t in range (ntypes) :
        ax.errorbar (np.arange (0, m+1, 1), np.mean (W_list[i][t,:,:], axis=0), 
                      1.96 * np.sqrt (np.var (W_list[i][t,:,:], axis=0) / nsims), capsize=5, markersize=4,
                      color=color_dict[scenarios[t]], marker='o')
    
    amp_list = list ()
    binom_coefficients = np.array ([math.comb (m, i) for i in range (m+1)]) / (m+1)
    
    nmax_d_beneficial = np.zeros (m+1)
    nmax_d_beneficial[1] = m
    
    nmax_deleterious = np.zeros (m+1)
    nmax_deleterious[0] = 1
    nmax_deleterious[2] = math.comb (m,2) / (m-1)
    for j in range (3,m+1) :
        nmax_deleterious[j] = math.comb (m,j) / (m+1)
    
    for j in range (3,m+1) :
        nmax_d_beneficial[j] = math.comb (m,j) / (m+1)

    amp_list.append (compute_Ma (np.mean (traj_list[i], axis=0)))
    amp_list.append (nmax_d_beneficial)
    amp_list.append (nmax_deleterious)
    amp_list.append (binom_coefficients)


    nmax_conditional = cp.deepcopy (binom_coefficients)
    nmax_conditional[-1] = 1
    nmax_conditional[-2] = 0
    amp_list.append (nmax_conditional)
    
    for a in range (len (amp_list)) :
        ax.plot (np.arange (0, m+1, 1), amp_list[a],
                  linestyle='-', color=dark_colors[a], markersize=5, zorder=10, linewidth=.5, marker='x')      
    
    ax.set_xticks ( np.arange (0, m+1, 1), np.arange (0, m+1, 1))
    ax.set_ylabel (r'$W_m$')
    if i == len (Ms) - 1 :
        ax.set_xlabel (r'distance, $m$')
    if i == 0 :
        ax.set_title ('C', weight='bold', loc='left', pad=3)

    ## W
    ax = axs[i,3]

    parts = ax.violinplot( [np.sum (W_list[i][t,:,:], axis=1) for t in range(ntypes)], positions=np.arange (0,ntypes,1),
                           widths=0.8, showmeans=True )

    # Color each violin
    for j, pc in enumerate(parts['bodies']):
        pc.set_facecolor(color_dict[scenarios[j]])
        pc.set_edgecolor('black')
        pc.set_alpha(0.7)

    # Median line styling
    parts['cmeans'].set_color('black')
    parts['cmeans'].set_linewidth(1.0)

    for key in ['cbars', 'cmins', 'cmaxes']:
        parts[key].set_color('black')
        parts[key].set_linewidth(0.5)
    
    for j in range (ntypes) :        
        ax.scatter (j, np.sum (amp_list[j]), marker='x', color=dark_colors[j], alpha=.75)   

    
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    ax.set_ylabel (r'$W$')
    ax.set_xticks(np.arange (0,ntypes,1))
    if i == len (Ms) - 1 :
        ax.set_xticklabels([x[:4] for x in scenarios], rotation=45)
    else :
        ax.set_xticklabels([])

    if i == 0 :
        ax.set_title ('D', weight='bold', loc='left', pad=3)
    
    ## P    
    ax = axs[i,4]

    parts = ax.violinplot( [P_list[i][t,:] for t in range(ntypes)], positions=np.arange (0,ntypes,1),
                           widths=0.8, showmeans=True )

    # Color each violin
    for j, pc in enumerate(parts['bodies']):
        pc.set_facecolor(color_dict[scenarios[j]])
        pc.set_edgecolor('black')
        pc.set_alpha(0.7)

    # Median line styling
    parts['cmeans'].set_color('black')
    parts['cmeans'].set_linewidth(1.0)

    for key in ['cbars', 'cmins', 'cmaxes']:
        parts[key].set_color('black')
        parts[key].set_linewidth(0.5)

    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    ax.set_ylabel (r'$P$')    
    ax.set_xticks(np.arange (0,ntypes,1))

    epaths = np.zeros (nsims)
    for j in range (nsims) :
        epaths[j] = compute_expected_paths ( traj_list[i][j,:] )
    
    exp_paths = [np.mean (epaths), 0, 0, 1/(m+1), 1]
    for j in range (ntypes) :        
        ax.scatter (j, exp_paths[j], marker='x', color=dark_colors[j], alpha=.75)    
    
    if i == len (Ms) - 1 :
        ax.set_xticklabels([x[:4] for x in scenarios], rotation=45)
    else :
        ax.set_xticklabels([])

    if i == 0 :
        ax.set_title ('E', weight='bold', loc='left', pad=3)    
    
    i += 1

# --- shared legend handles ---------------------------------------
# scenario colours (the 'o' errorbars common to all first 3 columns)
handles = [
    Line2D([0], [0], marker='o', linestyle='-', linewidth=1, markersize=4,
           color=color_dict[s], label=s)
    for s in scenarios
]
# optionally the theory curves from columns 0/1
handles += [
    Line2D([0], [0], color='black',  linestyle='dotted', linewidth=1, label='approx. theory'),
    Line2D([0], [0], marker='x', color='black', linestyle='-', linewidth=.5, label='theory'),
]

# --- position it under the first 3 columns -----------------------
fig.canvas.draw()                      # finalise constrained_layout positions
last      = len(Ms) - 1
box_left  = axs[last, 0].get_position()
box_right = axs[last, 2].get_position()
x_center  = 0.6 * (box_left.x0 + box_right.x1)

fig.legend(
    handles=handles,
    loc='upper center',
    bbox_to_anchor=(.525, box_left.y0 - 0.10),  # tune the -0.10 offset
    bbox_transform=fig.transFigure,
    #ncol=len(handles),
    frameon=False,
    ncol=5,
)
    
if save :
    plt.savefig (os.path.join (outputdir, 'figure_2.pdf'), bbox_inches='tight')
    plt.close ()
else :
    plt.show ()

2
4
2
6
2
3
4
